# Titanic Data Cleaning and Preprocessing

**Author:** Yuktha Ratna Puvvadi

This notebook examines the Titanic dataset, handles missing
values, engineers features, detects and removes extreme outliers, encodes
categories, standardizes numerical fields, and saves a modeling-ready CSV.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "titanic_preprocessing.py").exists():
    candidate = PROJECT_ROOT / "titanic-preprocessing"
    if candidate.exists():
        PROJECT_ROOT = candidate

sys.path.insert(0, str(PROJECT_ROOT))
import titanic_preprocessing as pipeline

DATA_PATH = PROJECT_ROOT / "data" / "Titanic-Dataset.csv"
df = pd.read_csv(DATA_PATH)
df.head()


## 1. Initial inspection

In [ ]:
print(f"Shape: {df.shape}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nMissing values:")
display(
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    }).sort_values("missing_count", ascending=False)
)


## 2. Missing-value plan

- **Age:** median within Sex x Pclass x Title groups; overall median fallback.
- **Embarked:** mode (`S`).
- **Cabin:** create `CabinKnown` and `Deck`; label missing decks `Unknown`.
- **Duplicate rows:** remove them before any fitted preprocessing.


In [ ]:
featured, imputation_details = pipeline.build_features(df)
display(pd.Series(imputation_details, name="decision"))
display(featured[[
    "PassengerId", "Sex", "Pclass", "Title", "Age", "Embarked",
    "Deck", "CabinKnown", "FamilySize", "TicketGroupSize", "FarePerPerson"
]].head())
print("Missing values remaining in engineered fields:", int(featured.isna().sum().sum()))


## 3. Outlier analysis

The standard 1.5x IQR rule is calculated for learning and reporting. It would
remove too many genuine high-fare passengers, so the final removal rule uses
3.0x IQR for only extreme Age or Fare values.


In [ ]:
filtered, removed, outlier_summary, selected_bounds = (
    pipeline.analyze_and_remove_outliers(featured)
)
display(outlier_summary)
print("Chosen bounds:", selected_bounds)
print("Extreme rows removed:", len(removed))
print("Rows retained:", len(filtered))


## 4. Encoding and standardization

In [ ]:
readable = pipeline.make_readable_dataset(filtered)
processed, scaler_parameters = pipeline.encode_and_standardize(readable)

print("Readable cleaned shape:", readable.shape)
print("Preprocessed shape:", processed.shape)
print("Final missing values:", int(processed.isna().sum().sum()))
display(scaler_parameters)
display(processed.head())


## 5. Recreate and save all project outputs

In [ ]:
summary = pipeline.run_pipeline()
display(pd.Series({
    "Input rows": summary["input_rows"],
    "Rows removed as extreme outliers": summary["extreme_outlier_rows_removed"],
    "Final rows": summary["final_rows"],
    "Final columns": summary["final_columns"],
    "Final missing values": summary["missing_values_after_total"],
}))
print("Files saved under:", PROJECT_ROOT / "outputs")


## 6. Visual checks

In [ ]:
for filename in [
    "01_missing_values_before.png",
    "02_outliers_before.png",
    "03_outliers_after.png",
    "04_target_balance_before_after.png",
    "05_feature_scaling_comparison.png",
]:
    print(filename)
    display(Image(filename=str(PROJECT_ROOT / "outputs" / "figures" / filename), width=760))


## Conclusion

The final `outputs/titanic_preprocessed.csv` contains no missing values and only
numerical features. The readable cleaned table, removed-row audit, quality
reports, scaler parameters, and preprocessing decisions are also saved.

For future model training, split the data first and fit learned preprocessing
steps only on the training set to avoid leakage.
